# ARC-AGI Solver — Colab Training

Trains a decoder-only transformer on a cohort of ARC tasks using RE-ARC synthetic data.
The model learns in-context: given K demonstration (input, output) pairs, predict the output
for a new input.

**To switch clusters:** change `ACTIVE` in Cell 6 and re-run from Cell 6 onwards.

**Cell order:**
1. Check GPU → 1b. Mount Drive → 2. Clone repo → 3. ARC data → 4. RE-ARC data
→ 5. Categories → 6. Config → 6b. Pre-tokenise → 7. Train → 8. View log
→ 9. Download checkpoint → 10. Evaluate (greedy / TTA / TTT) → 11. Resume

**Runtime:** A100 recommended (Pro). T4 works but is ~5× slower.

In [ ]:
# ── Cell 1: Check GPU ───────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected — training will be very slow on CPU.')

In [ ]:
# ── Cell 1b: Mount Google Drive ─────────────────────────────────────────────
# Checkpoints and pre-tokenised data are saved to Drive so they survive
# session disconnects.  First run: follow the authorisation link.
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_BASE          = '/content/drive/MyDrive/arc-agi'
DRIVE_CKPT_DIR      = f'{DRIVE_BASE}/checkpoints'
DRIVE_TOKENIZED_DIR = f'{DRIVE_BASE}/tokenized'

for d in (DRIVE_CKPT_DIR, DRIVE_TOKENIZED_DIR):
    Path(d).mkdir(parents=True, exist_ok=True)

print(f'Checkpoint dir:   {DRIVE_CKPT_DIR}')
print(f'Tokenized dir:    {DRIVE_TOKENIZED_DIR}')
print()
ckpts = sorted(Path(DRIVE_CKPT_DIR).glob('*.pt'))
print(f'Existing checkpoints ({len(ckpts)}):')
for p in ckpts:
    print(f'  {p.name}  ({p.stat().st_size/1e6:.1f} MB)')

In [ ]:
# ── Cell 2: Clone repo ──────────────────────────────────────────────────────
import os
from pathlib import Path

REPO        = 'arc-agi-solver'
GITHUB_USER = 'rodehyde'
REPO_ROOT   = f'/content/{REPO}'

if not Path(REPO_ROOT).exists():
    os.system(f'git clone https://github.com/{GITHUB_USER}/{REPO}.git {REPO_ROOT}')
else:
    os.system(f'git -C {REPO_ROOT} pull --ff-only')

os.chdir(REPO_ROOT)
print(f'Working directory: {os.getcwd()}')
os.system('git log --oneline -3')


In [ ]:
# ── Cell 3: Download ARC training data ──────────────────────────────────────
import io, urllib.request, zipfile
from pathlib import Path

DATA_DIR = Path('data')
if (DATA_DIR / 'training').exists() and len(list((DATA_DIR / 'training').glob('*.json'))) >= 400:
    print(f"ARC training data already present ({len(list((DATA_DIR/'training').glob('*.json')))} tasks).")
else:
    ARC_URL = 'https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip'
    print(f'Downloading ARC-AGI dataset...')
    with urllib.request.urlopen(ARC_URL) as r:
        raw = r.read()
    print(f'Downloaded {len(raw)/1e6:.1f} MB')
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        for split in ('training', 'evaluation'):
            dest = DATA_DIR / split
            dest.mkdir(exist_ok=True)
            members = [n for n in zf.namelist() if f'data/{split}/' in n and n.endswith('.json')]
            for m in members:
                (dest / Path(m).name).write_bytes(zf.read(m))
    print(f"training: {len(list((DATA_DIR/'training').glob('*.json')))} tasks")
    print(f"evaluation: {len(list((DATA_DIR/'evaluation').glob('*.json')))} tasks")

In [ ]:
# ── Cell 4: Download RE-ARC synthetic data ───────────────────────────────────
# RE-ARC provides 1,000 synthetic examples per task (used for training).
# Downloads ~110 MB zip and extracts all 400 task files.
from pathlib import Path
RE_ARC_DIR = Path('data/re_arc')
if RE_ARC_DIR.exists() and len(list(RE_ARC_DIR.glob('*.json'))) >= 400:
    print(f"RE-ARC already present ({len(list(RE_ARC_DIR.glob('*.json')))} files).")
else:
    import subprocess
    subprocess.run(['python', 'scripts/download_re_arc.py'], check=True)

In [ ]:
# ── Cell 4b: Download BARC synthetic data (optional — only for all_400_barc) ─
# BARC (~100k synthetic ARC-like problems) adds diverse training signal beyond
# the 400 original ARC tasks.  Only needed when ACTIVE='all_400_barc' or any
# config with n_barc > 0.  Safe to skip for ARC-only configs.
#
# Source: xu3kev/BARC on HuggingFace (~300 MB download)
# Output: data/barc/<id>.json  in standard ARC format (train pairs only)
# Time:   ~10 min on Colab (dominated by HuggingFace download + conversion)

import subprocess, sys
from pathlib import Path

BARC_DIR = Path('data/barc')

if BARC_DIR.exists() and len(list(BARC_DIR.glob('*.json'))) > 1000:
    print(f'BARC already present: {len(list(BARC_DIR.glob("*.json")))} tasks in {BARC_DIR}')
else:
    print('Downloading BARC from HuggingFace (xu3kev/BARC)...')
    result = subprocess.run(
        [sys.executable, 'scripts/download_barc.py'],
        capture_output=True, text=True,
    )
    out = result.stdout
    print(out[-3000:] if len(out) > 3000 else out)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])
    n = len(list(BARC_DIR.glob('*.json'))) if BARC_DIR.exists() else 0
    print(f'Done: {n} BARC tasks saved.')


In [ ]:
# ── Cell 5: Generate category labels ────────────────────────────────────────
import json, subprocess
from pathlib import Path
if not Path('results/categories_training.json').exists():
    print('Generating category labels...')
    subprocess.run(['python', '-m', 'src.explore'], check=True)
else:
    print('categories_training.json already present.')
data = json.loads(Path('results/categories_training.json').read_text())
su_tasks = [tid for tid, cats in data.items() if 'STRUCTURE_UNCHANGED' in cats]
print(f'STRUCTURE_UNCHANGED tasks: {len(su_tasks)}')

In [ ]:
# ── Cell 5b: Sanity check — EXTEND_RECT tasks ───────────────────────────────
import json
from pathlib import Path

cats = json.loads(Path('results/categories_training.json').read_text())
er_tasks = [tid for tid, task_cats in cats.items() if 'EXTEND_RECT' in task_cats]
print(f'EXTEND_RECT task count: {len(er_tasks)}')
print(f'Sample task IDs: {er_tasks[:5]}')

re_arc_dir = Path('data/re_arc')
missing = [tid for tid in er_tasks if not (re_arc_dir / f'{tid}.json').exists()]
if missing:
    print(f'WARNING: {len(missing)} tasks missing RE-ARC data: {missing[:10]}')
else:
    print(f'All {len(er_tasks)} EXTEND_RECT tasks have RE-ARC data in data/re_arc/')

In [ ]:
# ── Cell 6: Training configuration ──────────────────────────────────────────
# To run a different cluster: change ACTIVE to the desired key and
# re-run from this cell onwards (6b → 7 → ...).

ACTIVE = 'all_400_arc'   # ← only line you need to change

CONFIGS = {

    # ── all_400_arc: one model on ALL 400 ARC tasks (Step 2 + 3) ─────────────
    # Trains on all 400 ARC training tasks using LOO + D4/colour augmentation.
    # No RE-ARC or BARC download needed.  Larger model: 512-dim, 12 layers.
    # Val: leave-one-pair-out across ALL tasks (fair estimate of generalisation).
    'all_400_arc': dict(
        mode         = 'ALL',          # auto-discover every task in data/training/
        data_source  = 'arc',
        val_task_ids = [],             # training script uses all tasks for LOO val
        d_model      = 512,            # ↑ from 384 (Step 3: larger model)
        n_layers     = 12,             # ↑ from 8
        epochs          = 300,
        steps_per_epoch = 200,
        max_tokens      = 96000,
    ),

    # ── all_400_barc: all 400 ARC tasks + BARC synthetic data (Step 4) ────────
    # Same as all_400_arc but mixes in BARC synthetic tasks.
    # Requires BARC download cell to have been run first (Cell 4b).
    # n_barc = how many BARC tasks are randomly drawn and added to the pool.
    'all_400_barc': dict(
        mode         = 'ALL',
        data_source  = 'arc',
        val_task_ids = [],
        d_model      = 512,
        n_layers     = 12,
        n_barc       = 400,            # add 400 BARC tasks alongside 400 ARC tasks
        epochs          = 300,
        steps_per_epoch = 200,
        max_tokens      = 96000,
    ),

    # ── geometric_arc: geometric tasks trained on ORIGINAL ARC pairs only ────
    # Same 98 tasks as 'geometric' but uses --data-source arc (LOO + D4/colour
    # augmentation on original 2-10 ARC training pairs) instead of RE-ARC.
    # No RE-ARC download needed. Val uses ARC exact match by default.
    # Split: 89 train / 9 val
    'geometric_arc': dict(
        mode         = 'TASK_IDS',
        data_source  = 'arc',
        task_ids     = [
            '007bbfb7', '025d127b', '05f2a901', '0a938d79', '0dfd9992',
            '0e206a2e', '11852cab', '1b60fb0c', '1e0a9b12', '228f6490',
            '25ff71a9', '28bf18c6', '29ec7d0e', '321b1fc6', '3345333e',
            '3618c87e', '363442ee', '3906de3d', '3af2c5a8', '3c9b0459',
            '4290ef0e', '4522001f', '46f33fce', '496994bd', '49d1d64f',
            '4c4377d9', '4c5c2cf0', '508bd3b6', '5168d44c', '57aa92db',
            '6150a2bd', '62c24649', '67e8384a', '6855a6e4', '68b16354',
            '6a1e5592', '6aa20dc0', '6d0aefbc', '6fa7a44f', '73251a56',
            '7468f01a', '746b3537', '74dd1130', '760b3cac', '7b7f7511',
            '834ec97d', '846bdb03', '88a10436', '890034e9', '8be77c9e',
            '8e5a5113', '8eb1be9a', '9172f3a0', '93b581b8', '963e52fc',
            '98cf29f8', '9dfd6313', '9ecd008a', 'a1570a43', 'a3df8b1e',
            'a416b8f3', 'a79310a0', 'ae4f1146', 'b8825c91', 'b8cdaf2b',
            'bc1d5164', 'beb8660c', 'c3f564a4', 'c444b776', 'c59eb873',
            'c9e6f938', 'cce03e0d', 'ce602527', 'd13f3404', 'd8c310e9',
            'dc0a314f', 'dc433765', 'e3497940', 'e40b9e2f', 'e48d4e1a',
            'e5062a87', 'e76a88a6', 'e8dc4411', 'e98196ab', 'eb281b96',
            'f25fbde4', 'f25ffba3', 'f8a8fe49', 'f9012d9b',
        ],
        val_task_ids = [
            '017c7c7b', '1caeab9d', '2bcee788', '47c1f68c', '56dc2b01',
            '952a094c', '9d9215db', 'a78176bb', 'e21d9049',
        ],
        epochs          = 300,
        steps_per_epoch = 200,
        max_tokens      = 96000,
    ),


    # ── geometric: spatial-transformation tasks (98 tasks) ───────────────────
    'geometric': dict(
        mode         = 'TASK_IDS',
        task_ids     = [
            '007bbfb7', '025d127b', '05f2a901', '0a938d79', '0dfd9992',
            '0e206a2e', '11852cab', '1b60fb0c', '1e0a9b12', '228f6490',
            '25ff71a9', '28bf18c6', '29ec7d0e', '321b1fc6', '3345333e',
            '3618c87e', '363442ee', '3906de3d', '3af2c5a8', '3c9b0459',
            '4290ef0e', '4522001f', '46f33fce', '496994bd', '49d1d64f',
            '4c4377d9', '4c5c2cf0', '508bd3b6', '5168d44c', '57aa92db',
            '6150a2bd', '62c24649', '67e8384a', '6855a6e4', '68b16354',
            '6a1e5592', '6aa20dc0', '6d0aefbc', '6fa7a44f', '73251a56',
            '7468f01a', '746b3537', '74dd1130', '760b3cac', '7b7f7511',
            '834ec97d', '846bdb03', '88a10436', '890034e9', '8be77c9e',
            '8e5a5113', '8eb1be9a', '9172f3a0', '93b581b8', '963e52fc',
            '98cf29f8', '9dfd6313', '9ecd008a', 'a1570a43', 'a3df8b1e',
            'a416b8f3', 'a79310a0', 'ae4f1146', 'b8825c91', 'b8cdaf2b',
            'bc1d5164', 'beb8660c', 'c3f564a4', 'c444b776', 'c59eb873',
            'c9e6f938', 'cce03e0d', 'ce602527', 'd13f3404', 'd8c310e9',
            'dc0a314f', 'dc433765', 'e3497940', 'e40b9e2f', 'e48d4e1a',
            'e5062a87', 'e76a88a6', 'e8dc4411', 'e98196ab', 'eb281b96',
            'f25fbde4', 'f25ffba3', 'f8a8fe49', 'f9012d9b',
        ],
        val_task_ids = [
            '017c7c7b', '1caeab9d', '2bcee788', '47c1f68c', '56dc2b01',
            '952a094c', '9d9215db', 'a78176bb', 'e21d9049',
        ],
        epochs          = 300,
        steps_per_epoch = 200,
        max_tokens      = 96000,
    ),

    # ── C7: reflection / rotation (15 tasks) ─────────────────────────────────
    'C7_mirror': dict(
        mode         = 'TASK_IDS',
        task_ids     = [
            '3af2c5a8', '3c9b0459', '496994bd', '4c4377d9', '4c5c2cf0',
            '62c24649', '67e8384a', '68b16354', '6d0aefbc', '6fa7a44f',
            '7468f01a', 'a416b8f3',
        ],
        val_task_ids = [
            '760b3cac', '8be77c9e', 'c9e6f938',
        ],
        epochs          = 200,
        steps_per_epoch = 200,
        max_tokens      = 96000,
    ),

    # ── C8: compartment fill (curated) ───────────────────────────────────────
    'C8_compartment_fill': dict(
        mode         = 'TASK_IDS',
        task_ids     = [
            '09629e4f', '1190e5a7', '1e32b0e9', '272f95fa',
            '29623171', '6d0160f0', '941d9a10',
        ],
        val_task_ids = [
            '54d9e175', '6773b310',
        ],
        epochs          = 200,
        steps_per_epoch = 200,
        max_tokens      = 96000,
    ),

    # ── pattern_restoration ───────────────────────────────────────────────────
    'pattern_restoration': dict(
        mode         = 'TASK_IDS',
        task_ids     = [
            '9ecd008a', 'b8825c91', '29ec7d0e', '484b58aa', 'c3f564a4',
        ],
        val_task_ids = [
            '3345333e', '3631a71a',
        ],
        epochs          = 200,
        steps_per_epoch = 200,
        max_tokens      = 96000,
    ),

    # ── C3: mirror reflections ────────────────────────────────────────────────
    'C3_mirror': dict(
        mode         = 'TASK_IDS',
        task_ids     = [
            '3af2c5a8', '49d1d64f', '4c4377d9', '62c24649', '67e8384a',
            '6d0aefbc', '6fa7a44f', '8be77c9e', '8d5021e8', 'a416b8f3', 'c9e6f938',
        ],
        val_task_ids = [
            '7fe24cdd',
        ],
        epochs          = 100,
        steps_per_epoch = 200,
        max_tokens      = 96000,
    ),

}

# ── Shared hyperparameters (can be overridden per-config via d_model / n_layers) ─
K_CONTEXT     = 3
D_MODEL       = cfg.get('d_model',   384) if 'cfg' in dir() else 384   # resolved below
N_HEADS       = 8
N_LAYERS      = cfg.get('n_layers',  8)   if 'cfg' in dir() else 8
LR            = 3e-4
WARMUP_EPOCHS = 5
SAVE_EVERY    = 25
LOG_EVERY     = 1

# ── Resolve active config ─────────────────────────────────────────────────────
import json
from pathlib import Path

cfg             = CONFIGS[ACTIVE]
MODE            = cfg['mode']
DATA_SOURCE     = cfg.get('data_source', 'rearc')
TASK_IDS        = cfg.get('task_ids', [])
VAL_TASK_IDS    = cfg.get('val_task_ids', [])
CATEGORY        = cfg.get('category', '')
EPOCHS          = cfg['epochs']
STEPS_PER_EPOCH = cfg['steps_per_epoch']
MAX_TOKENS      = cfg['max_tokens']
N_BARC          = cfg.get('n_barc', 0)

# Per-config model size overrides
D_MODEL  = cfg.get('d_model',  384)
N_LAYERS = cfg.get('n_layers', 8)

if MODE == 'TASK_IDS':
    target_tasks = TASK_IDS
    run_tag      = ACTIVE
elif MODE == 'ALL':
    target_tasks = sorted(p.stem for p in Path('data/training').glob('*.json'))
    run_tag      = ACTIVE
else:
    cats_data    = json.loads(Path('results/categories_training.json').read_text())
    target_tasks = [tid for tid, cats in cats_data.items() if CATEGORY in cats]
    run_tag      = CATEGORY

LOG_FILE = f'results/train_{run_tag.lower()}.log'

print(f'Active:    {ACTIVE}')
print(f'Mode:      {MODE}')
print(f'Train tasks ({len(target_tasks)}): {target_tasks[:5]}{"..." if len(target_tasks) > 5 else ""}')
print(f'Val tasks   ({len(VAL_TASK_IDS)}): {VAL_TASK_IDS}')
print(f'Epochs:    {EPOCHS} × {STEPS_PER_EPOCH} steps = {EPOCHS*STEPS_PER_EPOCH:,} total')
print(f'd_model:   {D_MODEL},  heads: {N_HEADS},  layers: {N_LAYERS},  k_context: {K_CONTEXT}')
print(f'LR:        {LR},  warmup: {WARMUP_EPOCHS} epochs,  max_tokens: {MAX_TOKENS}')
print(f'Data source: {DATA_SOURCE}  |  n_barc: {N_BARC}')


In [ ]:
# ── Cell 6b: Pre-tokenise training tasks ────────────────────────────────────
# Encodes every RE-ARC example once to numpy arrays and saves to disk.
# Skipped automatically when data_source='arc' (no RE-ARC data used).
import subprocess, sys, shutil
from pathlib import Path

if DATA_SOURCE == 'arc':
    print(f'Data source is "arc" — skipping pre-tokenisation (RE-ARC not used).')
else:
    local_tok  = Path('data/tokenized')
    drive_tok  = Path(DRIVE_TOKENIZED_DIR)
    local_tok.mkdir(parents=True, exist_ok=True)

    # ── Restore any files already on Drive ──────────────────────────────────────
    restored = 0
    for p in drive_tok.glob('*.npz'):
        dest = local_tok / p.name
        if not dest.exists():
            shutil.copy2(p, dest)
            restored += 1
    if restored:
        print(f'Restored {restored} .npz file(s) from Drive.')

    # ── Tokenise anything still missing ─────────────────────────────────────────
    already = {p.stem for p in local_tok.glob('*.npz')}
    missing = [tid for tid in target_tasks if tid not in already]

    if not missing:
        print(f'All {len(target_tasks)} tasks already tokenised ({len(already)} total on disk).')
    else:
        print(f'Tokenising {len(missing)} tasks (already have {len(already)})...')
        result = subprocess.run(
            [sys.executable, 'scripts/pretokenize.py', '--tasks'] + missing,
            capture_output=True, text=True,
        )
        out = result.stdout
        print(out[-3000:] if len(out) > 3000 else out)
        if result.returncode != 0:
            print('STDERR:', result.stderr[-1000:])

    # ── Sync new files back to Drive ─────────────────────────────────────────────
    synced = 0
    for p in local_tok.glob('*.npz'):
        dest = drive_tok / p.name
        if not dest.exists():
            shutil.copy2(p, dest)
            synced += 1
    if synced:
        print(f'Saved {synced} new .npz file(s) to Drive.')
    print(f'Tokenized: {len(list(local_tok.glob("*.npz")))} tasks on disk.')


In [ ]:
# ── Cell 7: Run training ─────────────────────────────────────────────────────
# Checkpoints saved to Google Drive: DRIVE_CKPT_DIR/transformer_c{run_tag}_best.pt
#                                    DRIVE_CKPT_DIR/transformer_c{run_tag}_final.pt
#
# Checkpoint selection: best exact match on actual test pairs (all training pairs
# as context → predict test output → compare to known answer).  Logged as 'test acc/exact'.
import subprocess, sys

base_cmd = [
    sys.executable, '-u', 'scripts/train_transformer.py',
    '--epochs',          str(EPOCHS),
    '--steps-per-epoch', str(STEPS_PER_EPOCH),
    '--k-context',       str(K_CONTEXT),
    '--d-model',         str(D_MODEL),
    '--n-heads',         str(N_HEADS),
    '--n-layers',        str(N_LAYERS),
    '--lr',              str(LR),
    '--warmup-epochs',   str(WARMUP_EPOCHS),
    '--max-tokens',      str(MAX_TOKENS),
    '--max-seq-len',     '6000',
    '--save-every',      str(SAVE_EVERY),
    '--log-every',       str(LOG_EVERY),
    '--log',             LOG_FILE,
    '--run-name',        run_tag,
    '--ckpt-dir',        DRIVE_CKPT_DIR,   # ← save directly to Drive
    '--data-source',     DATA_SOURCE,
]

if N_BARC > 0:
    base_cmd += ['--n-barc', str(N_BARC)]

if MODE == 'TASK_IDS':
    cmd = base_cmd + ['--task-ids'] + target_tasks
elif MODE == 'ALL':
    cmd = base_cmd + ['--task-ids'] + target_tasks   # full list passed explicitly
else:
    cmd = base_cmd + ['--category', CATEGORY]

if VAL_TASK_IDS:
    cmd = cmd + ['--val-arc-task-ids'] + VAL_TASK_IDS

print('Command:', ' '.join(cmd[:10]), '...')
print(f'Checkpoints → {DRIVE_CKPT_DIR}')
print('=' * 60)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'Exit code: {proc.returncode}')


In [ ]:
# ── Cell 8: View training log ────────────────────────────────────────────────
from pathlib import Path
log = Path(LOG_FILE)
if log.exists():
    lines = log.read_text().splitlines()
    print(f'Log: {log} ({len(lines)} lines)')
    print('\n'.join(lines[-40:]))
else:
    print('Log file not found yet.')

In [ ]:
# ── Cell 9: Download best checkpoint to your local machine ──────────────────
# The checkpoint is already safe in Google Drive.
# Run this only if you also want a local copy (e.g. to run evaluate.py locally).
from pathlib import Path
from google.colab import files

ckpt = Path(DRIVE_CKPT_DIR) / f'transformer_c{run_tag}_best.pt'
if ckpt.exists():
    print(f'Downloading {ckpt.name}  ({ckpt.stat().st_size/1e6:.1f} MB)')
    files.download(str(ckpt))
else:
    print(f'Checkpoint not found: {ckpt}')
    print('Available checkpoints in Drive:')
    for p in sorted(Path(DRIVE_CKPT_DIR).glob('*.pt')):
        print(f'  {p.name}  ({p.stat().st_size/1e6:.1f} MB)')

In [ ]:
# ── Cell 10: Evaluate checkpoint (greedy / TTA / TTT) ───────────────────────
# Runs leave-one-out evaluation on the original ARC training pairs.
#
# mode='greedy' — single forward pass, argmax per cell
# mode='tta'    — (n_d4 × n_perms) combinations voted per cell
#                 n_d4=8: all D4 orientations (rotation+reflection); n_d4=1: colour-only
# mode='ttt'    — fine-tune on task training pairs (M steps) then run TTA
# mode='all'    — run all three and print a side-by-side comparison
#
# TTA expected improvement: greedy ~53% → tta ~70%+ exact match
# D4 geometric TTA gives an additional ~5-10% on top of colour-only TTA.
# TTT adds a further gain by adapting weights to the task's colours + geometry.

import subprocess, sys
from pathlib import Path

EVAL_MODE   = 'all'    # 'greedy' | 'tta' | 'ttt' | 'all'
N_PERMS     = 20       # colour permutations per D4 orientation
N_D4        = 8        # D4 orientations: 1=colour-only TTA, 8=all geometric symmetries
TTT_STEPS   = 50       # fine-tune steps for TTT
TTT_LR      = 1e-4     # TTT learning rate

ckpt_path = str(Path(DRIVE_CKPT_DIR) / f'transformer_c{run_tag}_best.pt')
cmd_eval = [
    sys.executable, '-u', 'scripts/evaluate.py',
    '--checkpoint', ckpt_path,
    '--mode',       EVAL_MODE,
    '--n-perms',    str(N_PERMS),
    '--n-d4',       str(N_D4),
    '--ttt-steps',  str(TTT_STEPS),
    '--ttt-lr',     str(TTT_LR),
    '--k-context',  str(K_CONTEXT),
    '--verbose',
]

print('Checkpoint:', ckpt_path)
print(f'Mode: {EVAL_MODE}  |  TTA: {N_D4} D4 orientations × {N_PERMS} colour perms = {N_D4*N_PERMS} votes')
print('=' * 60)

proc = subprocess.Popen(cmd_eval, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'Exit code: {proc.returncode}')


In [ ]:
# ── Cell 11 (optional): Resume from checkpoint ───────────────────────────────
# If the Colab session disconnected mid-training, run cells 1–6b to restore
# the environment (GPU, Drive, clone, data, config, pre-tokenise), then run
# this cell.  It rebuilds the training command from scratch.

import subprocess, sys
from pathlib import Path

RESUME_FROM = str(Path(DRIVE_CKPT_DIR) / f'transformer_c{run_tag}_best.pt')

base_cmd = [
    sys.executable, '-u', 'scripts/train_transformer.py',
    '--epochs',          str(EPOCHS),
    '--steps-per-epoch', str(STEPS_PER_EPOCH),
    '--k-context',       str(K_CONTEXT),
    '--d-model',         str(D_MODEL),
    '--n-heads',         str(N_HEADS),
    '--n-layers',        str(N_LAYERS),
    '--lr',              str(LR),
    '--warmup-epochs',   str(WARMUP_EPOCHS),
    '--max-tokens',      str(MAX_TOKENS),
    '--max-seq-len',     '6000',
    '--save-every',      str(SAVE_EVERY),
    '--log-every',       str(LOG_EVERY),
    '--log',             LOG_FILE,
    '--run-name',        run_tag,
    '--ckpt-dir',        DRIVE_CKPT_DIR,
    '--data-source',     DATA_SOURCE,
]

if N_BARC > 0:
    base_cmd += ['--n-barc', str(N_BARC)]

if MODE in ('TASK_IDS', 'ALL'):
    cmd_resume = base_cmd + ['--task-ids'] + target_tasks + ['--resume', RESUME_FROM]
else:
    cmd_resume = base_cmd + ['--category', CATEGORY] + ['--resume', RESUME_FROM]

if VAL_TASK_IDS:
    cmd_resume = cmd_resume + ['--val-arc-task-ids'] + VAL_TASK_IDS

print('Resuming from:', RESUME_FROM)
print('Command:', ' '.join(cmd_resume[:10]), '...')
print('=' * 60)

proc = subprocess.Popen(cmd_resume, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'Exit code: {proc.returncode}')
